# Phase 1 Final Split / Feature / Leakage Manifest Readiness

Notebook fix marker: `phase1_final_split_feature_leakage_plan_v1_20260421`

Purpose: record the final split, feature provenance and leakage-audit manifest contract required before final comparator runners are implemented. This notebook does **not** create final folds, extract final features, run final leakage audits, train models, or open headline claims.

Scientific integrity rule: final split/feature/leakage manifests must be generated by future final pipeline code from locked sources. Smoke feature rows and smoke audits cannot satisfy this readiness package.

## Technical Basis And Boundary

This notebook follows the current repository contract:

- `configs/phase1/final_comparator_artifacts.json`: final comparator manifests require `final_split_manifest`, `final_feature_manifest` and `final_leakage_audit`.
- `configs/phase1/final_split_feature_leakage.json`: defines split, feature provenance and leakage-audit schemas.
- `configs/split/loso_subject.yaml`: primary final split is leave-one-subject-out by `participant_id`.
- `src/phase1/final_split_feature_leakage.py`: fail-closed plan runner. It records missing manifests; it does not create final evidence.

Expected current result: split/feature/leakage manifests are missing, materialized final-scope signal audit remains a blocker, `claim_ready=false`, and no smoke artifacts are promoted.

In [ ]:
# Cell 1 - Runtime setup, Drive mount, clone/pull repo, and unit tests.
# This cell intentionally runs tests before writing split/feature/leakage readiness artifacts.

from pathlib import Path
import base64
import getpass
import subprocess

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752') if IN_COLAB else Path.cwd()

def run(cmd, cwd=None, check=True, redact_token=True):
    display = list(map(str, cmd))
    if redact_token:
        display = ['<redacted-token-command>' if 'http.extraHeader=Authorization:' in item else item for item in display]
    print('$', ' '.join(display))
    result = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def github_auth_args():
    if not IN_COLAB:
        return []
    token = getpass.getpass('Nhập GitHub token cho repo private; Enter nếu runtime đã có quyền: ')
    if not token:
        return []
    header = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return ['-c', f'http.extraHeader={header}']

GIT_AUTH_ARGS = github_auth_args()
if not REPO_DIR.exists():
    run(['git', *GIT_AUTH_ARGS, 'clone', REPO_URL, str(REPO_DIR)])
else:
    print(f'Repo exists: {REPO_DIR}')

run(['git', *GIT_AUTH_ARGS, 'pull', '--ff-only'], cwd=REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=False)
if unit_result.returncode != 0:
    print('Unit tests failed. Stop here; do not write split/feature/leakage readiness artifacts from an unclean codebase.')
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])
print('Unit tests passed; continuing to split/feature/leakage readiness checks.')

In [ ]:
# Cell 2 - Explicit source paths.
# Use the reviewed final comparator artifact plan from notebook 17. Do not silently follow latest.txt for claim-affecting planning.

from pathlib import Path

PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'
FINAL_COMPARATOR_ARTIFACT_RUN = DRIVE_ROOT / 'artifacts/phase1_final_comparator_artifact_plan/20260421T080131180467Z'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_split_feature_leakage_plan'
READINESS_CONFIG = REPO_DIR / 'configs/phase1/final_split_feature_leakage.json'

EXPECTED_LOCKED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
EXPECTED_COMPARATOR_ARTIFACT_STATUS = 'phase1_final_comparator_artifact_plan_recorded'
EXPECTED_REQUIRED_MANIFESTS = ['final_split_manifest', 'final_feature_manifest', 'final_leakage_audit']

print('Prereg bundle:', PREREG_BUNDLE)
print('Final comparator artifact run:', FINAL_COMPARATOR_ARTIFACT_RUN)
print('Readiness config:', READINESS_CONFIG)
print('Plan output root:', OUTPUT_ROOT)

In [ ]:
# Cell 3 - JSON/hash helpers used only for orchestration review.

import hashlib
import json
from pathlib import Path


def read_json(path: Path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding='utf-8'))


def sha256_file(path: Path) -> str:
    path = Path(path)
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def latest_run(root: Path) -> Path:
    pointer = Path(root) / 'latest.txt'
    if not pointer.exists():
        raise FileNotFoundError(pointer)
    return Path(pointer.read_text(encoding='utf-8').strip())

In [ ]:
# Cell 4 - Validate prereg identity and final comparator artifact boundary.

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_file_sha256 = sha256_file(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_LOCKED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

artifact_summary = read_json(FINAL_COMPARATOR_ARTIFACT_RUN / 'phase1_final_comparator_artifact_plan_summary.json')
artifact_claim_state = read_json(FINAL_COMPARATOR_ARTIFACT_RUN / 'phase1_final_comparator_claim_state.json')
artifact_contract = read_json(FINAL_COMPARATOR_ARTIFACT_RUN / 'phase1_final_comparator_artifact_contract.json')
print('Comparator artifact plan status:', artifact_summary.get('status'))
print('Comparator artifact claim_ready:', artifact_summary.get('claim_ready'))
print('Smoke metrics promoted:', artifact_summary.get('smoke_metrics_promoted'))
assert artifact_summary.get('status') == EXPECTED_COMPARATOR_ARTIFACT_STATUS
assert artifact_summary.get('claim_ready') is False
assert artifact_summary.get('headline_phase1_claim_open') is False
assert artifact_summary.get('smoke_metrics_promoted') is False
assert artifact_claim_state.get('full_phase1_claim_bearing_run_allowed') is False
for item in EXPECTED_REQUIRED_MANIFESTS:
    assert item in artifact_contract.get('required_artifacts_per_comparator', []), f'Missing manifest in artifact contract: {item}'

In [ ]:
# Cell 5 - Hash-link split/feature/leakage readiness source, config and notebook.

HASH_TARGETS = [
    'configs/phase1/final_split_feature_leakage.json',
    'configs/phase1/final_comparator_artifacts.json',
    'configs/split/loso_subject.yaml',
    'configs/data/snapshot.yaml',
    'src/phase1/final_split_feature_leakage.py',
    'src/phase1/final_comparator_artifacts.py',
    'src/cli.py',
    'tests/unit/test_phase1_final_split_feature_leakage.py',
    'notebooks/18_colab_phase1_final_split_feature_leakage_plan.ipynb',
]

hash_manifest = []
for rel in HASH_TARGETS:
    path = REPO_DIR / rel
    exists = path.exists()
    hash_manifest.append({'path': rel, 'exists': exists, 'sha256': sha256_file(path) if exists else None})
print(json.dumps(hash_manifest, indent=2))
assert all(item['exists'] for item in hash_manifest), 'All split/feature/leakage readiness source/config/notebook files must exist.'

## Expected Artifact Contract

The CLI command below writes a timestamped non-claim plan under `artifacts/phase1_final_split_feature_leakage_plan/`:

- `phase1_final_split_feature_leakage_plan_inputs.json`
- `phase1_final_split_feature_leakage_plan_summary.json`
- `phase1_final_split_feature_leakage_plan_report.md`
- `phase1_final_split_feature_leakage_contract.json`
- `phase1_final_split_manifest_readiness.json`
- `phase1_final_feature_manifest_readiness.json`
- `phase1_final_leakage_audit_readiness.json`
- `phase1_final_split_feature_leakage_source_links.json`
- `phase1_final_split_feature_leakage_missing_manifests.json`
- `phase1_final_split_feature_leakage_claim_state.json`
- `phase1_final_split_feature_leakage_implementation_plan.json`

Expected current interpretation: final split, feature and leakage manifests are missing; final-scope materialized signal audit is still a blocker; claims remain closed.

In [ ]:
# Cell 6 - Run the CLI-backed split/feature/leakage readiness plan.
# This command records schema/readiness only. It does not create final folds or features.

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_split_feature_leakage_plan',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--comparator-artifact-run', str(FINAL_COMPARATOR_ARTIFACT_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--readiness-config', 'configs/phase1/final_split_feature_leakage.json',
    '--artifact-config', 'configs/phase1/final_comparator_artifacts.json',
]
run(cmd, cwd=REPO_DIR)
print('Final split/feature/leakage readiness command completed. Review missing manifests before implementing final comparator runners.')

In [ ]:
# Cell 7 - Verify readiness artifacts and closeout state.

run_dir = latest_run(OUTPUT_ROOT)
print('Latest final split/feature/leakage readiness output:', run_dir)
required_files = [
    'phase1_final_split_feature_leakage_plan_inputs.json',
    'phase1_final_split_feature_leakage_plan_summary.json',
    'phase1_final_split_feature_leakage_plan_report.md',
    'phase1_final_split_feature_leakage_contract.json',
    'phase1_final_split_manifest_readiness.json',
    'phase1_final_feature_manifest_readiness.json',
    'phase1_final_leakage_audit_readiness.json',
    'phase1_final_split_feature_leakage_source_links.json',
    'phase1_final_split_feature_leakage_missing_manifests.json',
    'phase1_final_split_feature_leakage_claim_state.json',
    'phase1_final_split_feature_leakage_implementation_plan.json',
]
for name in required_files:
    exists = (run_dir / name).exists()
    print(f'{name} exists = {exists}')
    assert exists, f'Missing required artifact: {name}'

summary = read_json(run_dir / 'phase1_final_split_feature_leakage_plan_summary.json')
contract = read_json(run_dir / 'phase1_final_split_feature_leakage_contract.json')
split = read_json(run_dir / 'phase1_final_split_manifest_readiness.json')
feature = read_json(run_dir / 'phase1_final_feature_manifest_readiness.json')
leakage = read_json(run_dir / 'phase1_final_leakage_audit_readiness.json')
missing = read_json(run_dir / 'phase1_final_split_feature_leakage_missing_manifests.json')
claim_state = read_json(run_dir / 'phase1_final_split_feature_leakage_claim_state.json')

print(json.dumps({
    'run_dir': str(run_dir),
    'summary_status': summary.get('status'),
    'readiness_status': summary.get('readiness_status'),
    'claim_ready': summary.get('claim_ready'),
    'headline_phase1_claim_open': summary.get('headline_phase1_claim_open'),
    'full_phase1_claim_bearing_run_allowed': summary.get('full_phase1_claim_bearing_run_allowed'),
    'required_manifests': summary.get('required_manifests'),
    'all_required_manifests_present': summary.get('all_required_manifests_present'),
    'smoke_artifacts_promoted': summary.get('smoke_artifacts_promoted'),
    'schema_matches_comparator_artifact_plan': contract.get('schema_matches_comparator_artifact_plan'),
    'split_status': split.get('status'),
    'feature_status': feature.get('status'),
    'leakage_status': leakage.get('status'),
    'missing_manifests': missing.get('missing_manifests'),
    'blockers': claim_state.get('blockers'),
}, indent=2))

assert summary.get('status') == 'phase1_final_split_feature_leakage_plan_recorded'
assert summary.get('readiness_status') == 'planning_non_claim'
assert summary.get('claim_ready') is False
assert summary.get('headline_phase1_claim_open') is False
assert summary.get('full_phase1_claim_bearing_run_allowed') is False
assert summary.get('all_required_manifests_present') is False
assert summary.get('smoke_artifacts_promoted') is False
assert contract.get('schema_matches_comparator_artifact_plan') is True
assert split.get('status') == 'phase1_final_split_manifest_not_ready'
assert feature.get('status') == 'phase1_final_feature_manifest_not_ready'
assert leakage.get('status') == 'phase1_final_leakage_audit_not_ready'
assert feature.get('smoke_feature_rows_allowed_as_final') is False
assert leakage.get('required_schema', {}).get('outer_test_subject_in_teacher_fit_allowed') is False
assert leakage.get('required_schema', {}).get('test_time_privileged_or_teacher_outputs_allowed') is False
for expected_blocker in [
    'final_split_feature_leakage_plan_not_locked',
    'final_split_manifest_missing',
    'final_feature_manifest_missing',
    'final_leakage_audit_missing',
    'materialized_payload_signal_audit_for_final_scope_missing',
    'claim_blocked_until_final_split_feature_leakage_manifests_exist',
]:
    assert expected_blocker in claim_state.get('blockers', []), f'Missing blocker: {expected_blocker}'
print('\nNOT OK TO CLAIM: This final split/feature/leakage readiness plan does not prove decoder efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')

## Next Engineering Boundary

Allowed next implementation work: create the real final split manifest, final feature manifest and final leakage audit template from locked sources, then wire those manifests into future final comparator runners.

Not allowed: running claim-bearing final comparator fits, marking any comparator claim-evaluable, or using smoke feature rows/audits as final evidence before these manifests exist.